# 🚀 IRC 2026 — Autonomous Rover Object Detection (YOLOv8n)
## Unified Two-Class Benchmark: `0: cone` & `1: circle`

---

### 📌 Project Mission
This notebook cleans, harmonizes, balances, trains, evaluates, and exports an optimized object detection model for the **Indian Rover Challenge (IRC) 2026**.
The model detects two critical competition landmarks:
1. **`0: cone`** — Traffic cones and navigation markers (orange/red cones across varied lighting and angles).
2. **`1: circle`** — Circular markers, including normal colored circles, white chalk circles drawn on natural terrain/gravel, and chalk circles appearing alongside cones.

### ⚙️ System Specifications
- **Model**: Ultralytics YOLOv8 Nano (`yolov8n.pt`, ~3.2M params — real-time edge performance >40 FPS)
- **Framework**: PyTorch with CUDA 12+ & Automatic Mixed Precision (AMP / FP16)
- **Target Hardware**: Google Colab with NVIDIA Tesla T4 GPU (16 GB VRAM)
- **Storage Persistence**: Google Drive integration for raw dataset caching, automatic epoch checkpoint backups, and ONNX deployment weights.

---
### 🔒 Security Alert & API Key Hygiene
> ⚠️ **CRITICAL SECURITY NOTICE**: 
> In previous versions of this project (`circle(2).ipynb`), two Roboflow API keys were exposed in plain text:
> - `suneOGmh5Ob7BgYjwxOm`
> - `ZtaI3v5Ub65qYyRHsNBq`
> 
> **Action Required**: Revoke and rotate these API keys immediately in your [Roboflow Dashboard Settings](https://app.roboflow.com/).
> This notebook enforces zero-credential exposure using **Google Colab Secrets (`userdata.get('ROBOFLOW_API_KEY')`)** and masked interactive fallback (`getpass`). No keys are ever printed or written to disk.


---
## Section 1: NVIDIA T4 Hardware Acceleration & System Inspection

We verify the compute environment. Training YOLOv8n requires an NVIDIA GPU (NVIDIA Tesla T4 on Google Colab).
If no GPU is active, execution halts immediately with instructions to prevent slow CPU execution.


In [ ]:
# Step 1.1: Verify NVIDIA GPU Allocation
!nvidia-smi


In [ ]:
# Step 1.2: Install Required Production Dependencies
%pip install -q "ultralytics>=8.1.0" "roboflow>=1.1.0" "opencv-python-headless" "pyyaml" "matplotlib" "seaborn" "tqdm" "onnx>=1.14.0" "onnxslim>=0.1.20"


In [ ]:
# Step 1.3: Verify PyTorch, CUDA, and VRAM Availability
import sys
import torch
import ultralytics

print("=" * 70)
print("PYTORCH & CUDA SYSTEM CHECK")
print("=" * 70)
print(f"Python version      : {sys.version.split()[0]}")
print(f"PyTorch version     : {torch.__version__}")
print(f"Ultralytics version : {ultralytics.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ NO GPU DETECTED!\n\n"
        "To enable NVIDIA T4 in Google Colab:\n"
        "  1. Click 'Runtime' in the top menu.\n"
        "  2. Select 'Change runtime type'.\n"
        "  3. Under 'Hardware accelerator', select 'T4 GPU'.\n"
        "  4. Click 'Save' and re-run this notebook."
    )

gpu_device = 0
gpu_name = torch.cuda.get_device_name(gpu_device)
vram_total_gb = torch.cuda.get_device_properties(gpu_device).total_memory / (1024 ** 3)
cuda_capability = torch.cuda.get_device_capability(gpu_device)

print(f"Active GPU Device   : cuda:{gpu_device} ({gpu_name})")
print(f"Total VRAM          : {vram_total_gb:.2f} GB")
print(f"Compute Capability  : {cuda_capability[0]}.{cuda_capability[1]}")

# Allocate test tensor on CUDA device
test_tensor = torch.randn(10, 10, device=f"cuda:{gpu_device}")
print(f"CUDA Allocation Test: PASSED (Tensor device: {test_tensor.device})")
print("=" * 70)


---
## Section 2: Google Drive Mounting & Directory Architecture

We mount Google Drive to cache downloaded datasets, maintain checkpoints across Colab session disconnections, and save export artifacts.


In [ ]:
# Step 2.1: Mount Google Drive
from pathlib import Path
import os
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE_DRIVE = Path("/content/drive/MyDrive/IRC2026_Rover_Vision")
except ImportError:
    IN_COLAB = False
    BASE_DRIVE = Path("./IRC2026_Rover_Vision")

# Persistent storage on Drive
DRIVE_RAW_DATA = BASE_DRIVE / "datasets" / "raw"
DRIVE_MERGED_DATA = BASE_DRIVE / "datasets" / "merged"
DRIVE_MODELS = BASE_DRIVE / "models"
DRIVE_RESULTS = BASE_DRIVE / "results"

# High-speed local Colab SSD scratch directories
LOCAL_RAW_DIR = Path("/content/raw_datasets")
LOCAL_MERGED_DIR = Path("/content/IRC2026_UNIFIED_DATASET")
LOCAL_RUNS_DIR = Path("/content/runs/IRC2026")

for d in [DRIVE_RAW_DATA, DRIVE_MERGED_DATA, DRIVE_MODELS, DRIVE_RESULTS, LOCAL_RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Directory architecture initialized successfully.")
print(f"  Google Drive Base : {BASE_DRIVE}")
print(f"  Local Scratch Dir : {LOCAL_RAW_DIR}")
print(f"  Local Merged Dir  : {LOCAL_MERGED_DIR}")


---
## Section 3: Secure Dataset Ingestion & Roboflow Download with Drive Caching

We download the three target datasets specified for IRC 2026:
1. **Cone Detector**: `21-jyqut / cone-detector-uexft` (v1)
2. **Circle Shapes**: `deneme-3usbo / circle-9po6m` (v1)
3. **White Chalk Circle & Cone**: `kusuma-dv / white-chalk-circle-and-cone` (v3)

- **Colab Secrets**: Retrieved via `google.colab.userdata.get('ROBOFLOW_API_KEY')` with masked `getpass` fallback.
- **Drive Caching**: If already cached in Google Drive, archives are extracted instantly to local SSD, bypassing repeated network downloads.


In [ ]:
# Step 3.1: Retrieve Roboflow API Key securely
import getpass

ROBOFLOW_API_KEY = None

try:
    from google.colab import userdata
    try:
        ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        pass
except ImportError:
    pass

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    print("🔑 Please enter your Roboflow API Key (input will remain masked):")
    ROBOFLOW_API_KEY = getpass.getpass("ROBOFLOW_API_KEY: ")

if not ROBOFLOW_API_KEY or len(ROBOFLOW_API_KEY.strip()) == 0:
    raise ValueError("❌ Roboflow API Key is required to download datasets.")

print("✅ Roboflow API key acquired securely (Credential is masked).")


In [ ]:
# Step 3.2: Download or Restore the 3 Datasets
from roboflow import Roboflow
import zipfile

DATASETS_CONFIG = {
    "dataset_1_cone": {
        "workspace": "21-jyqut",
        "project": "cone-detector-uexft",
        "version": 1,
        "folder_name": "cone-detector-uexft-1"
    },
    "dataset_2_circle": {
        "workspace": "deneme-3usbo",
        "project": "circle-9po6m",
        "version": 1,
        "folder_name": "circle-9po6m-1"
    },
    "dataset_3_chalk": {
        "workspace": "kusuma-dv",
        "project": "white-chalk-circle-and-cone",
        "version": 3,
        "folder_name": "white-chalk-circle-and-cone-3"
    }
}

downloaded_paths = {}

for key, cfg in DATASETS_CONFIG.items():
    folder_name = cfg["folder_name"]
    local_target = LOCAL_RAW_DIR / folder_name
    drive_cache = DRIVE_RAW_DATA / f"{folder_name}.zip"

    print(f"\n------------------------------------------------------------")
    print(f"📦 Handling: {key} ({cfg['project']} v{cfg['version']})")
    print(f"------------------------------------------------------------")

    # Check 1: Already available on local SSD
    if local_target.exists() and (local_target / "data.yaml").exists():
        print(f"✅ Found locally on SSD: {local_target}")
        downloaded_paths[key] = local_target
        continue

    # Check 2: Available in Google Drive Cache
    if drive_cache.exists():
        print(f"⚡ Restoring from Google Drive cache: {drive_cache}")
        shutil.unpack_archive(str(drive_cache), str(local_target))
        downloaded_paths[key] = local_target
        print(f"✅ Extracted to local SSD: {local_target}")
        continue

    # Check 3: Download from Roboflow Universe
    print(f"🌐 Downloading from Roboflow Universe ({cfg['workspace']}/{cfg['project']} v{cfg['version']})...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    rf_project = rf.workspace(cfg["workspace"]).project(cfg["project"])
    rf_version = rf_project.version(cfg["version"])
    dl_res = rf_version.download("yolov8", location=str(local_target))
    downloaded_paths[key] = local_target

    # Cache dataset to Google Drive for future sessions
    print(f"💾 Caching dataset zip to Google Drive: {drive_cache}")
    shutil.make_archive(str(DRIVE_RAW_DATA / folder_name), 'zip', str(local_target))
    print(f"✅ Dataset cached to Google Drive successfully.")

print("\n" + "=" * 70)
print("ALL 3 DATASETS ARE READY ON HIGH-SPEED LOCAL SSD")
print("=" * 70)
for k, p in downloaded_paths.items():
    print(f"  {k:18s} -> {p}")


---
## Section 4: Deep Dataset Inspection & Dynamic Class Verification

Before altering any annotations, we perform a deep audit:
1. **Dynamic Class Verification**: Parse each dataset's `data.yaml` and verify raw class names against target classes:
   - `cone` $\\rightarrow$ `0: cone`
   - `circle` / `chalk` $\\rightarrow$ `1: circle`
2. **Diagnostic Scan**: Identify polygon annotations, out-of-bounds boxes, empty label files, orphaned files, and cross-split duplicate leaks.


In [ ]:
# Step 4.1: Dynamic Class Name & ID Verification from data.yaml
import yaml

TARGET_CLASSES = {0: "cone", 1: "circle"}
verified_class_mappings = {}

print("=" * 80)
print(f"{'DYNAMIC CLASS VERIFICATION FROM DATA.YAML':^80}")
print("=" * 80)

for d_key, d_path in downloaded_paths.items():
    yaml_file = d_path / "data.yaml"
    assert yaml_file.exists(), f"❌ data.yaml missing in {d_path}"
    with open(yaml_file, "r", encoding="utf-8") as f:
        meta = yaml.safe_load(f)

    raw_names = meta.get("names", [])
    if isinstance(raw_names, list):
        name_dict = {i: name for i, name in enumerate(raw_names)}
    elif isinstance(raw_names, dict):
        name_dict = {int(k): v for k, v in raw_names.items()}
    else:
        raise ValueError(f"Unrecognized names format in {yaml_file}: {raw_names}")

    mapping = {}
    print(f"\n📁 Dataset: {d_key} ({d_path.name})")
    print(f"   Declared Classes (nc={meta.get('nc')}): {name_dict}")

    for raw_id, name in name_dict.items():
        name_lower = name.lower().strip()
        if "cone" in name_lower and "circle" not in name_lower:
            target_id = 0
            target_name = "cone"
        elif "circle" in name_lower or "chalk" in name_lower:
            target_id = 1
            target_name = "circle"
        else:
            raise ValueError(f"❌ Ambiguous/unknown class '{name}' (ID {raw_id}) in {d_key}!")

        mapping[raw_id] = target_id
        print(f"   -> [Raw ID {raw_id}] '{name}' maps to [Target ID {target_id}] '{target_name}'")

    verified_class_mappings[d_key] = mapping

print("\n✅ All source classes successfully and dynamically mapped to {0: 'cone', 1: 'circle'}.")


In [ ]:
# Step 4.2: Comprehensive Dataset Diagnostic Scanner
import cv2
import hashlib
from collections import Counter, defaultdict

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}

def get_image_hash(path):
    """Computes SHA256 image content hash to detect exact duplicates."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def audit_dataset(dataset_name, dataset_path):
    splits = ["train", "valid", "test"]
    stats = {
        "dataset_name": dataset_name,
        "splits": {},
        "total_images": 0,
        "total_labels": 0,
        "total_boxes": 0,
        "class_counts": Counter(),
        "corrupted_images": 0,
        "empty_label_files": 0,
        "orphaned_images": 0,
        "orphaned_labels": 0,
        "polygon_annotations": 0,
        "out_of_bounds_boxes": 0,
        "zero_or_neg_size_boxes": 0,
        "image_hashes": {}
    }

    for split in splits:
        split_dir = dataset_path / split
        if not split_dir.exists() and split == "valid" and (dataset_path / "val").exists():
            split_dir = dataset_path / "val"
        if not split_dir.exists():
            continue

        img_dir = split_dir / "images" if (split_dir / "images").exists() else split_dir
        lbl_dir = split_dir / "labels" if (split_dir / "labels").exists() else split_dir

        image_files = {p.stem: p for p in img_dir.iterdir() if p.suffix in IMAGE_EXTENSIONS}
        label_files = {p.stem: p for p in lbl_dir.iterdir() if p.suffix == ".txt" and p.name != "classes.txt"}

        stats["total_images"] += len(image_files)
        stats["total_labels"] += len(label_files)
        stats["orphaned_images"] += len(set(image_files.keys()) - set(label_files.keys()))
        stats["orphaned_labels"] += len(set(label_files.keys()) - set(image_files.keys()))

        split_boxes = 0
        split_classes = Counter()

        for stem, img_path in image_files.items():
            fhash = get_image_hash(img_path)
            stats["image_hashes"][img_path] = (split, fhash)

            # Check readability
            try:
                img = cv2.imread(str(img_path))
                if img is None:
                    stats["corrupted_images"] += 1
            except Exception:
                stats["corrupted_images"] += 1

            lbl_path = label_files.get(stem)
            if not lbl_path:
                continue

            lines = lbl_path.read_text(encoding="utf-8", errors="replace").strip().splitlines()
            if not lines:
                stats["empty_label_files"] += 1
                continue

            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                try:
                    cls_id = int(float(parts[0]))
                except ValueError:
                    continue

                split_classes[cls_id] += 1
                stats["class_counts"][cls_id] += 1
                split_boxes += 1
                stats["total_boxes"] += 1

                if len(parts) == 5:
                    try:
                        xc, yc, w, h = map(float, parts[1:])
                        if w <= 0 or h <= 0:
                            stats["zero_or_neg_size_boxes"] += 1
                        if xc - w/2 < 0 or xc + w/2 > 1.0 or yc - h/2 < 0 or yc + h/2 > 1.0:
                            stats["out_of_bounds_boxes"] += 1
                    except ValueError:
                        pass
                elif len(parts) > 5:
                    stats["polygon_annotations"] += 1

        stats["splits"][split] = {
            "images": len(image_files),
            "labels": len(label_files),
            "boxes": split_boxes,
            "class_distribution": dict(split_classes)
        }

    return stats

audit_results = {}
for key, path in downloaded_paths.items():
    print(f"Auditing {key}...")
    audit_results[key] = audit_dataset(key, path)

print("✅ Pre-cleaning diagnostic scan complete.")


In [ ]:
# Step 4.3: Display Pre-Cleaning Diagnostic Report
print("=" * 85)
print(f"{'PRE-CLEANING DATASET AUDIT SUMMARY':^85}")
print("=" * 85)

for key, res in audit_results.items():
    print(f"\n📊 DATASET: {key}")
    print(f"   Root Path: {downloaded_paths[key]}")
    print(f"   Total Images: {res['total_images']} | Total Labels: {res['total_labels']} | Total Boxes: {res['total_boxes']}")
    print(f"   Raw Class Counts        : {dict(res['class_counts'])}")
    print(f"   Corrupted Images        : {res['corrupted_images']}")
    print(f"   Empty Label Files       : {res['empty_label_files']} (Negative background images)")
    print(f"   Polygon Annotations     : {res['polygon_annotations']} (Roboflow segmentation instances)")
    print(f"   Out-of-Bounds Boxes     : {res['out_of_bounds_boxes']} (Requires boundary clipping)")
    print(f"   Zero/Neg Area Boxes     : {res['zero_or_neg_size_boxes']}")

    for split_name, s_data in res["splits"].items():
        print(f"     - Split {split_name:6s} -> Images: {s_data['images']:4d} | Boxes: {s_data['boxes']:5d} | Classes: {s_data['class_distribution']}")

print("\n" + "=" * 85)


---
## Section 5: Data Harmonization, Leak Prevention & Train-Only Balancing

### Engineering Rules Enforced:
1. **Roboflow Polygon Conversion**: Polygons $(x_1, y_1, \dots, x_n, y_n)$ are converted into tight rectangular bounding boxes $[x_{\\min}, y_{\\min}, x_{\\max}, y_{\\max}] \\rightarrow [x_c, y_c, w, h]$.
2. **Safe Coordinate Boundary Clipping**: Out-of-bounds coordinates are strictly clipped to $[0.0, 1.0]$. Only zero-area or completely external boxes are dropped.
3. **Cross-Split Leakage Prevention**: We index SHA256 hashes of all images in `valid` and `test`. Any training image that duplicates a validation or test image is excluded from training to prevent data leakage.
4. **Train-Only Dataset Balancing**:
   - Cones have ~20,000 instances; circles have ~4,500 instances (~4.8:1 imbalance).
   - We balance the **TRAIN** split by oversampling circle-containing images (giving each duplicate a distinct filename) and subsampling pure-cone images until the ratio reaches $\\sim 1.2:1$.
   - **Validation and Test splits remain 100% UNTOUCHED** to guarantee fair, unbiased scientific evaluation.


In [ ]:
# Step 5.1: Safe Bounding Box & Polygon Annotation Conversion Engine
import math

def convert_and_clip_annotation(line, class_mapping):
    """
    Converts bbox or polygon line into normalized, clipped YOLO bbox.
    Returns: ((class_id, xc, yc, w, h), action_type) or (None, reject_reason)
    """
    parts = line.strip().split()
    if not parts or len(parts) < 5:
        return None, "empty_or_too_few_tokens"

    try:
        raw_class = int(float(parts[0]))
    except ValueError:
        return None, "bad_class"

    if raw_class not in class_mapping:
        return None, "unmapped_class"

    target_class = class_mapping[raw_class]

    try:
        coords = [float(x) for x in parts[1:]]
    except ValueError:
        return None, "bad_coordinates"

    if not all(math.isfinite(c) for c in coords):
        return None, "non_finite_coordinate"

    was_polygon = False
    was_clipped = False

    # Case A: Standard YOLO Bounding Box [xc, yc, w, h]
    if len(coords) == 4:
        xc, yc, w, h = coords
        if w <= 0 or h <= 0:
            return None, "zero_or_negative_dimensions"
        x1 = xc - w / 2.0
        y1 = yc - h / 2.0
        x2 = xc + w / 2.0
        y2 = yc + h / 2.0
    # Case B: Segmentation Polygon [x1, y1, x2, y2, ... xn, yn]
    else:
        if len(coords) < 6 or len(coords) % 2 != 0:
            return None, "malformed_polygon_vertex_pairs"
        was_polygon = True
        xs = coords[0::2]
        ys = coords[1::2]
        x1, y1 = min(xs), min(ys)
        x2, y2 = max(xs), max(ys)
        if x2 <= x1 or y2 <= y1:
            return None, "inverted_polygon_bounds"

    if x1 < 0.0 or y1 < 0.0 or x2 > 1.0 or y2 > 1.0:
        was_clipped = True

    # Safe boundary clipping
    cx1 = max(0.0, min(1.0, x1))
    cy1 = max(0.0, min(1.0, y1))
    cx2 = max(0.0, min(1.0, x2))
    cy2 = max(0.0, min(1.0, y2))

    new_w = cx2 - cx1
    new_h = cy2 - cy1

    if new_w < 1e-4 or new_h < 1e-4:
        return None, "completely_outside_image"

    new_xc = (cx1 + cx2) / 2.0
    new_yc = (cy1 + cy2) / 2.0

    action = "polygon_converted" if was_polygon else ("clipped" if was_clipped else "clean")
    return (target_class, new_xc, new_yc, new_w, new_h), action


In [ ]:
# Step 5.2: Execute Cross-Split Leakage Prevention, Harmonization & Train Balancing
if LOCAL_MERGED_DIR.exists():
    shutil.rmtree(LOCAL_MERGED_DIR)

for split in ["train", "valid", "test"]:
    (LOCAL_MERGED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (LOCAL_MERGED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

# 1. Collect SHA256 hashes of valid and test images to prevent train data leakage
eval_image_hashes = {}
for d_key, d_path in downloaded_paths.items():
    for split in ["valid", "test"]:
        s_dir = d_path / split
        if not s_dir.exists() and split == "valid" and (d_path / "val").exists():
            s_dir = d_path / "val"
        if not s_dir.exists():
            continue
        img_dir = s_dir / "images" if (s_dir / "images").exists() else s_dir
        for img_p in img_dir.iterdir():
            if img_p.suffix in IMAGE_EXTENSIONS:
                eval_image_hashes[get_image_hash(img_p)] = (d_key, split, img_p.name)

# 2. Stage images and sanitize annotations
train_catalog = []
sanitization_stats = Counter()

for d_key, d_path in downloaded_paths.items():
    class_map = verified_class_mappings[d_key]
    prefix = d_key.replace("dataset_", "")

    for split in ["train", "valid", "test"]:
        s_dir = d_path / split
        if not s_dir.exists() and split == "valid" and (d_path / "val").exists():
            s_dir = d_path / "val"
        if not s_dir.exists():
            continue

        img_dir = s_dir / "images" if (s_dir / "images").exists() else s_dir
        lbl_dir = s_dir / "labels" if (s_dir / "labels").exists() else s_dir

        for img_p in sorted(img_dir.iterdir()):
            if img_p.suffix not in IMAGE_EXTENSIONS:
                continue

            fhash = get_image_hash(img_p)

            # Leakage Check: Exclude train image if it duplicates any valid or test image
            if split == "train" and fhash in eval_image_hashes:
                sanitization_stats["train_leaks_prevented"] += 1
                continue

            lbl_p = lbl_dir / f"{img_p.stem}.txt"
            sanitized_boxes = []

            if lbl_p.exists():
                lines = lbl_p.read_text(encoding="utf-8", errors="replace").splitlines()
                for line in lines:
                    converted, action = convert_and_clip_annotation(line, class_map)
                    if converted:
                        sanitized_boxes.append(converted)
                        sanitization_stats[action] += 1
                    else:
                        sanitization_stats["boxes_dropped"] += 1

            unique_filename = f"{prefix}_{img_p.name}"

            # Valid and Test splits remain completely untouched
            if split != "train":
                dest_img = LOCAL_MERGED_DIR / split / "images" / unique_filename
                dest_lbl = LOCAL_MERGED_DIR / split / "labels" / f"{dest_img.stem}.txt"
                shutil.copy2(img_p, dest_img)
                lbl_lines = [f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}" for b in sanitized_boxes]
                dest_lbl.write_text("\n".join(lbl_lines) + ("\n" if lbl_lines else ""))
                sanitization_stats["eval_images_untouched"] += 1
            else:
                train_catalog.append({
                    "src_img": img_p,
                    "unique_name": unique_filename,
                    "boxes": sanitized_boxes,
                    "has_circle": any(b[0] == 1 for b in sanitized_boxes),
                    "has_cone": any(b[0] == 0 for b in sanitized_boxes),
                    "is_bg": len(sanitized_boxes) == 0
                })

# 3. Compute pre-balance training statistics
raw_train_cones = sum(sum(1 for b in item["boxes"] if b[0] == 0) for item in train_catalog)
raw_train_circles = sum(sum(1 for b in item["boxes"] if b[0] == 1) for item in train_catalog)

# 4. Apply Train-Only Balancing Algorithm (Target ratio: 1.2:1)
circle_items = [it for it in train_catalog if it["has_circle"]]
pure_cone_items = [it for it in train_catalog if it["has_cone"] and not it["has_circle"]]
bg_items = [it for it in train_catalog if it["is_bg"]]

# Oversample minority circle images (2x replication)
balanced_train_items = []
for it in circle_items:
    balanced_train_items.append(it)
    dup = dict(it)
    p = Path(it["unique_name"])
    dup["unique_name"] = f"{p.stem}_baldup1{p.suffix}"
    balanced_train_items.append(dup)

# Target cone count based on post-oversampled circles
post_circle_boxes = sum(sum(1 for b in it["boxes"] if b[0] == 1) for it in balanced_train_items)
desired_cone_boxes = int(post_circle_boxes * 1.25)

# Subsample pure cone images to reach desired cone count
accumulated_cones = sum(sum(1 for b in it["boxes"] if b[0] == 0) for it in balanced_train_items)
for it in pure_cone_items:
    c_count = sum(1 for b in it["boxes"] if b[0] == 0)
    if accumulated_cones + c_count <= desired_cone_boxes:
        balanced_train_items.append(it)
        accumulated_cones += c_count

# Include ~6% background negative images
num_bg = min(len(bg_items), max(10, int(len(balanced_train_items) * 0.06)))
balanced_train_items.extend(bg_items[:num_bg])

# Write balanced train split to disk
for item in balanced_train_items:
    dest_img = LOCAL_MERGED_DIR / "train" / "images" / item["unique_name"]
    dest_lbl = LOCAL_MERGED_DIR / "train" / "labels" / f"{dest_img.stem}.txt"
    shutil.copy2(item["src_img"], dest_img)
    lbl_lines = [f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}" for b in item["boxes"]]
    dest_lbl.write_text("\n".join(lbl_lines) + ("\n" if lbl_lines else ""))

# 5. Compute post-balance training statistics
balanced_train_cones = sum(sum(1 for b in item["boxes"] if b[0] == 0) for item in balanced_train_items)
balanced_train_circles = sum(sum(1 for b in item["boxes"] if b[0] == 1) for item in balanced_train_items)

print("=" * 80)
print(f"{'DATASET HARMONIZATION & BALANCING SUMMARY':^80}")
print("=" * 80)
print(f"Sanitization Statistics:")
for k, v in sanitization_stats.items():
    print(f"  {k:26s}: {v}")

print(f"\nTRAIN SET CLASS BALANCING RESULTS:")
print(f"  Pre-Balancing  -> Cone: {raw_train_cones:5d} | Circle: {raw_train_circles:5d} | Ratio: {raw_train_cones/max(1, raw_train_circles):.2f}:1")
print(f"  Post-Balancing -> Cone: {balanced_train_cones:5d} | Circle: {balanced_train_circles:5d} | Ratio: {balanced_train_cones/max(1, balanced_train_circles):.2f}:1")
print(f"  Total Balanced Train Images: {len(balanced_train_items)}")
print(f"  Validation & Test Images   : {sanitization_stats['eval_images_untouched']} (Strictly Untouched)")
print("=" * 80)


In [ ]:
# Step 5.3: Generate Unified data.yaml
data_yaml_content = f"""# IRC 2026 Autonomous Rover Object Detection Dataset
# Unified Two-Class Benchmark: 0: cone, 1: circle

path: {LOCAL_MERGED_DIR}
train: train/images
val: valid/images
test: test/images

nc: 2
names:
  0: cone
  1: circle
"""

unified_yaml_path = LOCAL_MERGED_DIR / "data.yaml"
unified_yaml_path.write_text(data_yaml_content, encoding="utf-8")
print(f"✅ Generated unified data.yaml at: {unified_yaml_path}")


In [ ]:
# Step 5.4: Generate Before-and-After Class Distribution Graphs
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = ["Class 0: Cone", "Class 1: Circle"]
colors = ["#ff7f0e", "#1f77b4"]

# Plot 1: Raw (Pre-balancing)
raw_counts = [raw_train_cones, raw_train_circles]
sns.barplot(x=categories, y=raw_counts, ax=axes[0], palette=colors)
axes[0].set_title(f"Raw Train Distribution (Imbalanced {raw_train_cones/max(1, raw_train_circles):.1f}:1)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Bounding Box Count")
for i, v in enumerate(raw_counts):
    axes[0].text(i, v + max(raw_counts)*0.02, str(v), ha='center', fontweight='bold')

# Plot 2: Balanced (Post-balancing)
balanced_counts = [balanced_train_cones, balanced_train_circles]
sns.barplot(x=categories, y=balanced_counts, ax=axes[1], palette=colors)
axes[1].set_title(f"Balanced Train Distribution (Optimized {balanced_train_cones/max(1, balanced_train_circles):.2f}:1)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Bounding Box Count")
for i, v in enumerate(balanced_counts):
    axes[1].text(i, v + max(balanced_counts)*0.02, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


---
## Section 6: Post-Merge Verification & Visual Ground-Truth Inspection

We execute strict mathematical checks:
- Exact 100% 1:1 image-to-label pairing.
- Zero out-of-bounds coordinates ($0.0 \\le x, y, w, h \\le 1.0$).
- Strictly `{0, 1}` class IDs.
- Visual inspection rendering sample images with ground-truth bounding boxes.


In [ ]:
# Step 6.1: Mathematical Assertion Checks
print("=" * 80)
print(f"{'MERGED DATASET VALIDATION REPORT':^80}")
print("=" * 80)

for split in ["train", "valid", "test"]:
    img_dir = LOCAL_MERGED_DIR / split / "images"
    lbl_dir = LOCAL_MERGED_DIR / split / "labels"

    imgs = sorted(list(img_dir.glob("*.*")))
    lbls = sorted(list(lbl_dir.glob("*.txt")))

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    assert img_stems == lbl_stems, f"❌ Discrepancy between images and labels in {split} split!"

    split_boxes = 0
    split_classes = Counter()

    for lbl_p in lbls:
        lines = lbl_p.read_text(encoding="utf-8").strip().splitlines()
        for line in lines:
            parts = line.strip().split()
            assert len(parts) == 5, f"❌ Malformed line in {lbl_p}: {line}"
            cls_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
            assert cls_id in (0, 1), f"❌ Invalid class ID {cls_id} in {lbl_p}"
            assert 0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0, f"❌ Out of bounds center in {lbl_p}"
            assert 0.0 < w <= 1.0 and 0.0 < h <= 1.0, f"❌ Out of bounds size in {lbl_p}"

            split_boxes += 1
            split_classes[cls_id] += 1

    print(f"Split {split.upper():6s} -> Images: {len(imgs):5d} | Bounding Boxes: {split_boxes:6d}")
    print(f"  -> Class 0 (cone)   : {split_classes[0]:5d}")
    print(f"  -> Class 1 (circle) : {split_classes[1]:5d}")

print("=" * 80)
print("✅ STRICT DATASET VALIDATION PASSED 100%!")


In [ ]:
# Step 6.2: Visual Sample Verification
import random

def visualize_annotated_samples(split="train", num_samples=6):
    img_dir = LOCAL_MERGED_DIR / split / "images"
    lbl_dir = LOCAL_MERGED_DIR / split / "labels"

    all_images = sorted(list(img_dir.glob("*.*")))
    sample_images = random.sample(all_images, min(num_samples, len(all_images)))

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    axes = axes.flatten()
    colors = {0: (255, 128, 0), 1: (0, 220, 255)}  # Orange for cone, Cyan for circle

    for idx, img_path in enumerate(sample_images):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h_img, w_img, _ = img.shape

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            lines = lbl_path.read_text(encoding="utf-8").strip().splitlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    xc, yc, w, h = map(float, parts[1:])
                    x1 = int((xc - w / 2) * w_img)
                    y1 = int((yc - h / 2) * h_img)
                    x2 = int((xc + w / 2) * w_img)
                    y2 = int((yc + h / 2) * h_img)

                    color = colors.get(cls_id, (0, 255, 0))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    label_name = TARGET_CLASSES.get(cls_id, str(cls_id))
                    cv2.putText(img, label_name, (x1, max(20, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        axes[idx].imshow(img)
        axes[idx].set_title(f"{img_path.name[:25]}...\n({split.upper()})", fontsize=10)
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

visualize_annotated_samples(split="train", num_samples=6)


---
## Section 7: YOLOv8n Training Pipeline (NVIDIA T4 Optimized)

### Training Features:
- **Periodic Google Drive Sync**: Automatic callback backs up `last.pt` and `best.pt` to Google Drive every 5 epochs and on completion.
- **Resumption Capability**: If a Colab session disconnects, training can be resumed directly from `last.pt` without losing progress.
- **T4 Tuning**: `batch=16`, `imgsz=640`, `device=0`, `amp=True` (FP16 Tensor Cores), `workers=2`.


In [ ]:
# Step 7.1: Setup Training Callbacks & Resumption Logic
from ultralytics import YOLO

# Check if a previous checkpoint exists on Drive to resume from
drive_last_pt = DRIVE_MODELS / "IRC2026_YOLOv8n_last.pt"
RESUME_TRAINING = False

if drive_last_pt.exists():
    print(f"🔄 Found existing checkpoint on Google Drive: {drive_last_pt}")
    print("Set RESUME_TRAINING = True below if you wish to continue training.")
else:
    print("✨ Starting fresh training run from official pretrained yolov8n.pt")

# Callback to sync checkpoints directly to Google Drive
def on_train_epoch_end_sync(trainer):
    """Syncs last.pt and best.pt to Google Drive every 5 epochs."""
    if (trainer.epoch + 1) % 5 == 0:
        local_last = trainer.wdir / "last.pt"
        local_best = trainer.wdir / "best.pt"
        if local_last.exists():
            shutil.copy2(local_last, DRIVE_MODELS / "IRC2026_YOLOv8n_last.pt")
        if local_best.exists():
            shutil.copy2(local_best, DRIVE_MODELS / "IRC2026_YOLOv8n_best.pt")
        print(f"\n💾 [Drive Sync] Epoch {trainer.epoch + 1} checkpoints safely backed up to Google Drive.")


In [ ]:
# Step 7.2: Launch YOLOv8n Training on NVIDIA T4
if RESUME_TRAINING and drive_last_pt.exists():
    print(f"🚀 Resuming training from {drive_last_pt}...")
    model = YOLO(str(drive_last_pt))
    train_results = model.train(resume=True)
else:
    print("🚀 Initializing official pretrained YOLOv8n...")
    model = YOLO("yolov8n.pt")
    model.add_callback("on_train_epoch_end", on_train_epoch_end_sync)

    TRAIN_CONFIG = {
        "data": str(unified_yaml_path),
        "epochs": 30,
        "imgsz": 640,
        "batch": 16,
        "device": 0,         # Force NVIDIA GPU
        "amp": True,         # Automatic Mixed Precision
        "workers": 2,        # Colab 2 vCPUs
        "patience": 15,      # Early stopping
        "optimizer": "auto",
        "project": str(LOCAL_RUNS_DIR),
        "name": "YOLOv8n_Cone_Circle_Unified",
        "exist_ok": True,
        "save": True,
        "save_period": 5,
        "plots": True,
        "verbose": True
    }

    train_results = model.train(**TRAIN_CONFIG)

print("✅ Training complete!")


---
## Section 8: Model Evaluation & Independent Test Set Benchmark

We inspect training curves, confusion matrix, and execute an unbiased evaluation on the **unseen test split**.


In [ ]:
# Step 8.1: Display Training Metric Plots & Confusion Matrix
from IPython.display import Image, display

run_dir = LOCAL_RUNS_DIR / "YOLOv8n_Cone_Circle_Unified"

plot_files = [
    ("Training Summary Curves", run_dir / "results.png"),
    ("Confusion Matrix", run_dir / "confusion_matrix.png"),
    ("F1-Confidence Curve", run_dir / "BoxF1_curve.png"),
    ("Precision-Recall Curve", run_dir / "BoxPR_curve.png")
]

for title, path in plot_files:
    if path.exists():
        print(f"\n📈 {title}:")
        display(Image(filename=str(path), width=850))
    else:
        print(f"ℹ️ {title} plot not found at {path}")


In [ ]:
# Step 8.2: Strict Evaluation on the Independent Test Split
best_weight_path = run_dir / "weights" / "best.pt"
if not best_weight_path.exists():
    best_weight_path = DRIVE_MODELS / "IRC2026_YOLOv8n_best.pt"

if best_weight_path.exists():
    best_model = YOLO(str(best_weight_path))
    print("=" * 70)
    print("EVALUATING BEST CHECKPOINT ON UNSEEN TEST SPLIT")
    print("=" * 70)
    test_metrics = best_model.val(
        data=str(unified_yaml_path),
        split="test",
        device=0,
        imgsz=640,
        batch=16,
        plots=True
    )
    print("\n🏆 TEST SPLIT PERFORMANCE:")
    print(f"  Overall mAP@50     : {test_metrics.box.map50:.4f}")
    print(f"  Overall mAP@50-95  : {test_metrics.box.map:.4f}")
    print(f"  Cone mAP@50 (0)    : {test_metrics.box.maps[0]:.4f}")
    print(f"  Circle mAP@50 (1)  : {test_metrics.box.maps[1]:.4f}")
    print("=" * 70)


---
## Section 9: Persistent Archival to Google Drive


In [ ]:
# Step 9.1: Backup final weights and session artifacts
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
drive_session_dir = DRIVE_RESULTS / f"run_{timestamp}"
drive_session_dir.mkdir(parents=True, exist_ok=True)

best_src = run_dir / "weights" / "best.pt"
last_src = run_dir / "weights" / "last.pt"

if best_src.exists():
    shutil.copy2(best_src, DRIVE_MODELS / "IRC2026_YOLOv8n_best.pt")
    shutil.copy2(best_src, DRIVE_MODELS / f"IRC2026_YOLOv8n_best_{timestamp}.pt")
    print(f"✅ Saved Best Weights to: {DRIVE_MODELS / 'IRC2026_YOLOv8n_best.pt'}")

if last_src.exists():
    shutil.copy2(last_src, DRIVE_MODELS / "IRC2026_YOLOv8n_last.pt")
    print(f"✅ Saved Last Weights to: {DRIVE_MODELS / 'IRC2026_YOLOv8n_last.pt'}")

shutil.copytree(run_dir, drive_session_dir / "training_run", dirs_exist_ok=True)
shutil.copy2(unified_yaml_path, DRIVE_MODELS / "IRC2026_data.yaml")
print(f"✅ Full run artifacts backed up to: {drive_session_dir}")


---
## Section 10: Interactive Inference & ONNX Rover Deployment Export

We test visual inference and export the trained model to ONNX for onboard rover deployment (ROS2 / TensorRT).


In [ ]:
# Step 10.1: Visual Inference on Random Test Split Images
best_model_path = DRIVE_MODELS / "IRC2026_YOLOv8n_best.pt"
if not best_model_path.exists():
    best_model_path = run_dir / "weights" / "best.pt"

model = YOLO(str(best_model_path))

test_images = sorted(list((LOCAL_MERGED_DIR / "test" / "images").glob("*.*")))
sample_test_images = random.sample(test_images, min(6, len(test_images)))

pred_results = model.predict(
    source=[str(p) for p in sample_test_images],
    imgsz=640,
    conf=0.25,
    iou=0.45,
    device=0,
    save=False
)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (res, img_p) in enumerate(zip(pred_results, sample_test_images)):
    plotted_rgb = res.plot()
    axes[idx].imshow(plotted_rgb)
    axes[idx].set_title(f"Pred: {img_p.name[:25]}...\n{len(res.boxes)} Detections", fontsize=10)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Step 10.2: Export to ONNX for Onboard Rover Inference (ROS2 / Jetson)
import onnx

print("=" * 70)
print("EXPORTING TRAINED BEST MODEL TO ONNX")
print("=" * 70)

onnx_file = model.export(
    format="onnx",
    dynamic=False,
    simplify=True,
    opset=12,
    imgsz=640
)

if onnx_file and os.path.exists(onnx_file):
    # Verify ONNX model integrity
    onnx_model = onnx.load(onnx_file)
    onnx.checker.check_model(onnx_model)
    print("✅ ONNX Model Checker: PASSED (Graph is structurally valid)")

    drive_onnx = DRIVE_MODELS / "IRC2026_YOLOv8n.onnx"
    shutil.copy2(onnx_file, drive_onnx)
    print(f"✅ ONNX exported and saved to Google Drive: {drive_onnx}")
    print("   Ready for TensorRT acceleration or ROS2 YOLO node on the rover.")
print("=" * 70)


---
# SECTION 11 — CUSTOM IMAGE UPLOAD & OBJECT DETECTION TESTING

### 🎯 Objective
After training and validating the YOLOv8n model, upload your own field images directly into Google Colab (from rover test drives, phone cameras, or competition runs) to verify whether the model correctly detects:
- **`cone` (Class 0)** — Traffic cones and navigation markers.
- **`circle` (Class 1)** — Normal circles and faint white chalk circles on ground terrain.
- **Both classes appearing together** in the same field of view.

Uses the best-trained `best.pt` checkpoint on CUDA (NVIDIA T4).


In [ ]:
# Step 11.1: Load Best-Trained Model & Verify Classes
from pathlib import Path
import os
import torch
from ultralytics import YOLO

print("=" * 75)
print("SECTION 11.1 — LOADING TRAINED YOLOv8n MODEL")
print("=" * 75)

# Automatically locate best.pt across standard local and Google Drive locations
candidate_paths = [
    Path("/content/runs/IRC2026/YOLOv8n_Cone_Circle_Unified/weights/best.pt"),
    Path("/content/drive/MyDrive/IRC2026_Rover_Vision/models/IRC2026_YOLOv8n_best.pt"),
    Path("/content/drive/MyDrive/IRC2026_YOLO_V5/models/best.pt"),
    Path("/content/drive/MyDrive/IRC2026_YOLO_V4/models/IRC2026_YOLOv8n_V1_best.pt")
]

best_model_path = None
for cp in candidate_paths:
    if cp.exists():
        best_model_path = cp
        break

# Fallback: search locally in /content for any best.pt
if best_model_path is None:
    found = list(Path("/content").rglob("*best.pt"))
    if found:
        best_model_path = found[0]

if best_model_path is None or not best_model_path.exists():
    raise FileNotFoundError(
        "❌ Trained checkpoint 'best.pt' not found!\n"
        "Please ensure Section 7 (Training) has run and produced model weights."
    )

print(f"📍 Located Best Checkpoint: {best_model_path}")
print(f"   Checkpoint Size        : {os.path.getsize(best_model_path) / (1024**2):.2f} MB")

# Check device availability
inference_device = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"⚡ Compute Device          : CUDA:0 ({gpu_name}, {vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ Compute Device          : CPU (CUDA not available)")

# Load model weights
trained_model = YOLO(str(best_model_path))

# Verify expected classes
model_classes = trained_model.names
print(f"🏷️  Loaded Model Classes    : {model_classes}")

assert len(model_classes) >= 2, f"❌ Model expected at least 2 classes, got {model_classes}"
assert "cone" in str(model_classes[0]).lower(), f"❌ Class 0 should be 'cone', got {model_classes[0]}"
assert "circle" in str(model_classes[1]).lower(), f"❌ Class 1 should be 'circle', got {model_classes[1]}"

print("✅ Model verification PASSED: Correct weights, CUDA acceleration & class mappings confirmed.")
print("=" * 75)


In [ ]:
# Step 11.2: Upload Custom Field Images
from pathlib import Path
import os
import shutil
from google.colab import files

UPLOAD_DIR = Path("/content/custom_rover_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

print("=" * 75)
print("SECTION 11.2 — UPLOAD CUSTOM IMAGES FOR INFERENCE TESTING")
print("=" * 75)
print("📤 Click 'Choose Files' below to upload test images.")
print("   Supported formats: JPG, JPEG, PNG, WEBP\n")

uploaded_files = files.upload()

uploaded_image_paths = []
if uploaded_files:
    for filename, content in uploaded_files.items():
        ext = Path(filename).suffix.lower()
        if ext in ALLOWED_EXTENSIONS:
            save_path = UPLOAD_DIR / filename
            with open(save_path, "wb") as f:
                f.write(content)
            uploaded_image_paths.append(save_path)
        else:
            print(f"⚠️ Skipping non-image file: {filename}")

if uploaded_image_paths:
    print(f"\n✅ Successfully uploaded {len(uploaded_image_paths)} image(s):")
    for idx, p in enumerate(uploaded_image_paths, 1):
        size_kb = os.path.getsize(p) / 1024
        print(f"   [{idx}] {p.name} ({size_kb:.1f} KB)")
else:
    print("⚠️ No images uploaded yet. Run this cell again to upload field images.")
print("=" * 75)


In [ ]:
# Step 11.3: Run Object Detection Inference
from pathlib import Path

# Configurable detection hyperparameters (modify as needed)
CONFIDENCE_THRESHOLD = 0.25  # Minimum confidence threshold
IOU_THRESHOLD = 0.45         # NMS IoU threshold
IMAGE_SIZE = 640             # Inference resolution

print("=" * 75)
print("SECTION 11.3 — RUNNING OBJECT DETECTION INFERENCE")
print("=" * 75)
print("⚙️ Inference Parameters:")
print(f"   - CONFIDENCE_THRESHOLD : {CONFIDENCE_THRESHOLD:.2f}")
print(f"   - IOU_THRESHOLD        : {IOU_THRESHOLD:.2f}")
print(f"   - IMAGE_SIZE           : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   - DEVICE               : {inference_device}")
print("-" * 75)

detection_records = []

if not uploaded_image_paths:
    print("⚠️ No uploaded images to test. Please run Cell 11.2 first.")
else:
    # Run batch prediction
    predict_results = trained_model.predict(
        source=[str(p) for p in uploaded_image_paths],
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE,
        device=inference_device,
        save=False,
        verbose=False
    )

    for img_path, res in zip(uploaded_image_paths, predict_results):
        extracted_detections = []
        if res.boxes is not None and len(res.boxes) > 0:
            for b in res.boxes:
                cls_id = int(b.cls[0].item())
                class_name = trained_model.names.get(cls_id, f"class_{cls_id}")
                conf_score = float(b.conf[0].item())
                x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())

                extracted_detections.append({
                    "class_id": cls_id,
                    "class_name": class_name,
                    "confidence": conf_score,
                    "bbox": (x1, y1, x2, y2)
                })

        detection_records.append({
            "image_path": img_path,
            "image_name": img_path.name,
            "detections": extracted_detections,
            "predict_result": res
        })

    print(f"✅ Inference complete for {len(detection_records)} image(s).")
    total_objs = sum(len(r["detections"]) for r in detection_records)
    print(f"   Total objects detected across all images: {total_objs}")
print("=" * 75)


In [ ]:
# Step 11.4: Annotated Image Visualization (Original vs Annotated)
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Distinct high-contrast colors for each class
CLASS_COLORS_BGR = {
    0: (0, 140, 255),    # Vibrant Orange for Cone (BGR)
    1: (255, 210, 0)     # Sky Blue / Cyan for Circle (BGR)
}

def annotate_image_opencv(img_bgr, detections):
    """Draws clear bounding boxes, class names, and confidence percentage."""
    canvas = img_bgr.copy()
    h, w, _ = canvas.shape

    font_scale = max(0.5, min(w, h) / 900.0)
    line_thickness = max(2, int(min(w, h) / 300.0))

    for det in detections:
        cls_id = det["class_id"]
        cls_name = det["class_name"]
        conf = det["confidence"]
        x1, y1, x2, y2 = map(int, det["bbox"])

        color = CLASS_COLORS_BGR.get(cls_id, (0, 255, 0))

        # 1. Draw rectangular bounding box
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, line_thickness)

        # 2. Format label: e.g. "cone 95.42%" or "circle 91.83%"
        label_text = f"{cls_name} {conf * 100:.2f}%"

        # 3. Label tag with solid background for readability
        (tw, th), baseline = cv2.getTextSize(
            label_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, max(1, line_thickness - 1)
        )
        tag_y1 = max(0, y1 - th - baseline - 6)
        tag_y2 = y1
        tag_x2 = min(w, x1 + tw + 6)

        cv2.rectangle(canvas, (x1, tag_y1), (tag_x2, tag_y2), color, -1)
        cv2.putText(
            canvas,
            label_text,
            (x1 + 3, tag_y2 - baseline - 2),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (0, 0, 0),
            max(1, line_thickness - 1),
            lineType=cv2.LINE_AA
        )

    return canvas

annotated_images_store = {}

if not detection_records:
    print("⚠️ No detection records to display. Run Cells 11.2 & 11.3 first.")
else:
    print("=" * 75)
    print("SECTION 11.4 — VISUALIZING ORIGINAL VS ANNOTATED DETECTION IMAGES")
    print("=" * 75)

    for rec in detection_records:
        img_p = rec["image_path"]
        orig_bgr = cv2.imread(str(img_p))
        if orig_bgr is None:
            continue

        orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
        annotated_bgr = annotate_image_opencv(orig_bgr, rec["detections"])
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

        annotated_images_store[rec["image_name"]] = {
            "orig_bgr": orig_bgr,
            "annotated_bgr": annotated_bgr,
            "orig_rgb": orig_rgb,
            "annotated_rgb": annotated_rgb
        }

        # Render side-by-side: Original vs Annotated
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        axes[0].imshow(orig_rgb)
        axes[0].set_title(f"Original: {rec['image_name']}", fontsize=12, fontweight="bold")
        axes[0].axis("off")

        det_count = len(rec["detections"])
        axes[1].imshow(annotated_rgb)
        axes[1].set_title(f"Annotated ({det_count} detected): {rec['image_name']}",
                          fontsize=12, fontweight="bold",
                          color="darkgreen" if det_count > 0 else "darkred")
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:
# Step 11.5: Per-Image Detection Breakdown & Metrics
import pandas as pd
from IPython.display import display

print("=" * 75)
print("SECTION 11.5 — PER-IMAGE DETECTION BREAKDOWN & METRICS")
print("=" * 75)

if not detection_records:
    print("⚠️ No detection records to summarize.")
else:
    for idx, rec in enumerate(detection_records, 1):
        img_name = rec["image_name"]
        dets = rec["detections"]

        cone_dets = [d for d in dets if d["class_id"] == 0]
        circle_dets = [d for d in dets if d["class_id"] == 1]

        total_count = len(dets)
        cone_count = len(cone_dets)
        circle_count = len(circle_dets)

        print(f"\n{'─' * 70}")
        print(f"Image name: {img_name}")
        print(f"{'─' * 70}")
        print(f"Total objects detected: {total_count}")
        print(f"Cones detected: {cone_count}")
        print(f"Circles detected: {circle_count}")

        if total_count > 0:
            print("\nDetection details:")
            table_rows = []
            cone_idx = 1
            circle_idx = 1

            for d in dets:
                if d["class_id"] == 0:
                    obj_name = f"Cone {cone_idx}"
                    cone_idx += 1
                else:
                    obj_name = f"Circle {circle_idx}"
                    circle_idx += 1

                table_rows.append({
                    "Object": obj_name,
                    "Confidence": f"{d['confidence'] * 100:.2f}%",
                    "Bounding Box (x1, y1, x2, y2)": f"({d['bbox'][0]:.0f}, {d['bbox'][1]:.0f}, {d['bbox'][2]:.0f}, {d['bbox'][3]:.0f})"
                })

            df_details = pd.DataFrame(table_rows)
            display(df_details)

            all_confs = [d["confidence"] for d in dets]
            cone_confs = [d["confidence"] for d in cone_dets]
            circle_confs = [d["confidence"] for d in circle_dets]

            avg_cone = f"{sum(cone_confs)/len(cone_confs)*100:.2f}%" if cone_confs else "N/A"
            avg_circle = f"{sum(circle_confs)/len(circle_confs)*100:.2f}%" if circle_confs else "N/A"
            high_conf = f"{max(all_confs)*100:.2f}%"
            low_conf = f"{min(all_confs)*100:.2f}%"

            print(f"Summary Metrics for {img_name}:")
            print(f"  • Total cone count          : {cone_count}")
            print(f"  • Total circle count        : {circle_count}")
            print(f"  • Average confidence (Cone) : {avg_cone}")
            print(f"  • Average confidence (Circle): {avg_circle}")
            print(f"  • Highest confidence        : {high_conf}")
            print(f"  • Lowest confidence         : {low_conf}")
        else:
            print("⚠️ No objects detected at or above confidence threshold.")

print("=" * 75)


In [ ]:
# Step 11.6: Save Annotated Images to Google Drive & Download Archive
from pathlib import Path
import shutil
import zipfile
import cv2
from google.colab import files

# Google Drive Target Directory Structure
DRIVE_V5_BASE = Path("/content/drive/MyDrive/IRC2026_YOLO_V5/inference")
DRIVE_ORIG_DIR = DRIVE_V5_BASE / "original"
DRIVE_ANNOT_DIR = DRIVE_V5_BASE / "annotated"

# Local Scratch Directory Structure
LOCAL_V5_BASE = Path("/content/IRC2026_YOLO_V5/inference")
LOCAL_ORIG_DIR = LOCAL_V5_BASE / "original"
LOCAL_ANNOT_DIR = LOCAL_V5_BASE / "annotated"

for d in [DRIVE_ORIG_DIR, DRIVE_ANNOT_DIR, LOCAL_ORIG_DIR, LOCAL_ANNOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("SECTION 11.6 — SAVING ANNOTATED IMAGES TO GOOGLE DRIVE")
print("=" * 75)

saved_annotated_files = []

for rec in detection_records:
    img_name = rec["image_name"]
    stem = Path(img_name).stem
    ext = Path(img_name).suffix

    annot_filename = f"{stem}_annotated{ext}"

    if img_name in annotated_images_store:
        orig_bgr = annotated_images_store[img_name]["orig_bgr"]
        annot_bgr = annotated_images_store[img_name]["annotated_bgr"]

        # 1. Save to local storage
        loc_orig = LOCAL_ORIG_DIR / img_name
        loc_annot = LOCAL_ANNOT_DIR / annot_filename
        cv2.imwrite(str(loc_orig), orig_bgr)
        cv2.imwrite(str(loc_annot), annot_bgr)
        saved_annotated_files.append(loc_annot)

        # 2. Save to Google Drive
        try:
            drv_orig = DRIVE_ORIG_DIR / img_name
            drv_annot = DRIVE_ANNOT_DIR / annot_filename
            cv2.imwrite(str(drv_orig), orig_bgr)
            cv2.imwrite(str(drv_annot), annot_bgr)
        except Exception as e:
            print(f"ℹ️ Google Drive write note for {img_name}: {e}")

print(f"✅ Saved {len(saved_annotated_files)} image(s) to:")
print(f"   Google Drive Annotated Dir : {DRIVE_ANNOT_DIR}")
print(f"   Google Drive Original Dir  : {DRIVE_ORIG_DIR}")
print(f"   Local Annotated Dir        : {LOCAL_ANNOT_DIR}")

# Create ZIP archive for one-click downloading from Colab
if saved_annotated_files:
    zip_path = Path("/content/IRC2026_Annotated_Detections.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for p in saved_annotated_files:
            zipf.write(p, arcname=p.name)

    print(f"\n📦 Created download archive: {zip_path.name} ({zip_path.stat().st_size / 1024:.1f} KB)")
    print("⬇️ Triggering browser download of annotated images...")
    try:
        files.download(str(zip_path))
    except Exception as e:
        print(f"ℹ️ In automated/headless mode, archive is ready at: {zip_path}")
print("=" * 75)


In [ ]:
# Step 11.7: Overall Detection Testing Report & CSV Export
import pandas as pd
from pathlib import Path
from IPython.display import display
from google.colab import files

print("=" * 75)
print("SECTION 11.7 — OVERALL DETECTION TESTING REPORT")
print("=" * 75)

report_rows = []
all_cone_confs = []
all_circle_confs = []

for rec in detection_records:
    img_name = rec["image_name"]
    dets = rec["detections"]

    cone_dets = [d for d in dets if d["class_id"] == 0]
    circle_dets = [d for d in dets if d["class_id"] == 1]

    cone_confs = [d["confidence"] for d in cone_dets]
    circle_confs = [d["confidence"] for d in circle_dets]
    all_confs = [d["confidence"] for d in dets]

    all_cone_confs.extend(cone_confs)
    all_circle_confs.extend(circle_confs)

    report_rows.append({
        "Image Name": img_name,
        "Total Detections": len(dets),
        "Cones Detected": len(cone_dets),
        "Circles Detected": len(circle_dets),
        "Average Confidence": f"{sum(all_confs)/len(all_confs)*100:.2f}%" if all_confs else "0.00%",
        "Cone Avg Conf": f"{sum(cone_confs)/len(cone_confs)*100:.2f}%" if cone_confs else "N/A",
        "Circle Avg Conf": f"{sum(circle_confs)/len(circle_confs)*100:.2f}%" if circle_confs else "N/A",
        "Confidence Scores": ", ".join([f"{c*100:.1f}%" for c in all_confs]) if all_confs else "None"
    })

df_report = pd.DataFrame(report_rows)

total_images_tested = len(detection_records)
total_cones_found = sum(r["Cones Detected"] for r in report_rows)
total_circles_found = sum(r["Circles Detected"] for r in report_rows)
all_detected_confs = all_cone_confs + all_circle_confs
overall_avg_conf = f"{sum(all_detected_confs)/len(all_detected_confs)*100:.2f}%" if all_detected_confs else "0.00%"

print("📊 AGGREGATE TESTING SUMMARY:")
print(f"   • Total Images Tested       : {total_images_tested}")
print(f"   • Total Cones Detected      : {total_cones_found}")
print(f"   • Total Circles Detected    : {total_circles_found}")
print(f"   • Total Landmarks Detected  : {total_cones_found + total_circles_found}")
print(f"   • Overall Average Confidence: {overall_avg_conf}")
print("-" * 75)

display(df_report)

# Export detection summary report as CSV
csv_drive_path = Path("/content/drive/MyDrive/IRC2026_YOLO_V5/inference/detection_summary.csv")
csv_local_path = Path("/content/IRC2026_YOLO_V5/inference/detection_summary.csv")

csv_drive_path.parent.mkdir(parents=True, exist_ok=True)
csv_local_path.parent.mkdir(parents=True, exist_ok=True)

df_report.to_csv(str(csv_local_path), index=False)
try:
    df_report.to_csv(str(csv_drive_path), index=False)
    print(f"\n💾 Saved summary report to Google Drive: {csv_drive_path}")
except Exception as e:
    print(f"ℹ️ Google Drive CSV note: {e}")

print(f"💾 Saved summary report locally        : {csv_local_path}")

print("⬇️ Triggering download of detection_summary.csv...")
try:
    files.download(str(csv_local_path))
except Exception as e:
    print(f"ℹ️ In automated/headless mode, CSV is ready at: {csv_local_path}")
print("=" * 75)
